Structure of the data file system: 

data/
 ├── serial-lt-hsc/
 │    ├── well_001/
 │    │     ├── 00d00h00m.tif
 │    │     ├── 00d03h00m.tif
 │    │     └── ...
 │    └── ...
 └── others/
      ├── well_101/
      │     ├── 00d00h00m.tif
      │     └── ...


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from sklearn.metrics import balanced_accuracy_score, recall_score, f1_score, balanced_accuracy_score, recall_score, f1_score, precision_recall_curve, average_precision_score,roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import Compose, ToTensor, Lambda, RandomChoice
import timm
import umap
from umap import UMAP
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

BASE = Path("/path/to/your/data/folder")
CLASSES = ["serial-lt-hsc", "others"]
LABEL_MAP = {"serial-lt-hsc": 0, "others": 1}
POS_IDX = 1
IMG_SZ = 128
BATCH_SIZE = 4
EPOCHS = 40
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on", DEVICE)

class GELU(nn.Module):
    def forward(self, x):
        return F.gelu(x)

ACTIVATION = nn.ReLU()


In [ ]:
angles = [90, 180, 270]
flip_ops = [
    Lambda(lambda img: img.transpose(Image.FLIP_LEFT_RIGHT)),
    Lambda(lambda img: img.transpose(Image.FLIP_TOP_BOTTOM)),
]
contrast_ops = [
    Lambda(lambda img, f=f: ImageEnhance.Contrast(img).enhance(f))
    for f in [0.8, 1.2, 1.5]
]
bright_ops = [
    Lambda(lambda img, f=f: ImageEnhance.Brightness(img).enhance(f))
    for f in [0.8, 1.2, 1.5]
]
all_augs = (
    [Lambda(lambda img, a=a: img.rotate(a)) for a in angles]
    + flip_ops
    + contrast_ops
    + bright_ops
)

class ResizeShorterSide:
    def __init__(self, short_side):
        self.short_side = short_side

    def __call__(self, img: Image.Image):
        w, h = img.size
        if w <= h:
            new_w = self.short_side
            new_h = int(h * self.short_side / w)
        else:
            new_h = self.short_side
            new_w = int(w * self.short_side / h)
        return img.resize((new_w, new_h), resample=Image.BILINEAR)

train_tfms = Compose([
    RandomChoice(all_augs),
    ResizeShorterSide(IMG_SZ),
    ToTensor(),
])
val_tfms = Compose([
    ResizeShorterSide(IMG_SZ),
    ToTensor(),
])


In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, folders, label_map, transform):
        self.folders = folders
        self.labels = [label_map[folder.parent.name] for folder in folders]
        self.transform = transform

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, index):
        folder = self.folders[index]
        paths = sorted(folder.glob("*.tif"))
        if not paths:
            raise ValueError(f"Empty sequence folder: {folder}")

        frames = [self.transform(Image.open(path).convert("RGB")) for path in paths]
        diffs = [torch.zeros_like(frames[0])]
        for prev, curr in zip(frames, frames[1:]):
            diffs.append(torch.abs(curr - prev))

        seq = torch.stack([
            torch.cat((frame, diff), dim=0) for frame, diff in zip(frames, diffs)
        ], dim=0)
        return seq, seq.size(0), self.labels[index]


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    seqs, lengths, labels = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True)
    return padded, lengths, labels

def has_tifs(folder: Path) -> bool:
    return any(folder.glob("*.tif"))

all_series = []
for cls in CLASSES:
    for d in (BASE / cls).iterdir():
        if d.is_dir():
            all_series.append(d)

labels = [LABEL_MAP[d.parent.name] for d in all_series]
train_idx, val_idx = train_test_split(
    list(range(len(all_series))),
    test_size=0.2,
    stratify=labels,
    random_state=42,
)
train_folders = [all_series[i] for i in train_idx if has_tifs(all_series[i])]
val_folders = [all_series[i] for i in val_idx if has_tifs(all_series[i])]

train_ds = SequenceDataset(train_folders, LABEL_MAP, train_tfms)
val_ds = SequenceDataset(val_folders, LABEL_MAP, val_tfms)
print(f"Train series: {len(train_ds)} | Val series: {len(val_ds)}")

train_labels = [LABEL_MAP[d.parent.name] for d in train_folders]
counts = np.bincount(train_labels)
weights = [(1.0 / counts[label]) for label in train_labels]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=0,
)
val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)


In [ ]:
class FrameEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model("convnext_tiny", pretrained=True, features_only=False)
        stem_conv = base.stem[0]
        new_conv = nn.Conv2d(6, 96, kernel_size=4, stride=4, padding=0, bias=False)
        new_conv.weight.data[:, :3, :, :] = stem_conv.weight.data
        new_conv.weight.data[:, 3:, :, :] = stem_conv.weight.data
        base.stem[0] = new_conv
        self.backbone = base
        self.out_dim = base.num_features

    def forward(self, x):
        features = self.backbone.forward_features(x)
        return features.mean(dim=(2, 3))


class BiLSTMSeqClassifier(nn.Module):
    def __init__(self, hidden=256, n_heads=4):
        super().__init__()
        self.enc = FrameEncoder()
        self.lstm = nn.LSTM(
            self.enc.out_dim, hidden, num_layers=2, bidirectional=True, batch_first=True
        )
        self.mha = nn.MultiheadAttention(
            embed_dim=hidden * 2, num_heads=n_heads, batch_first=True
        )
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 128),
            ACTIVATION,
            nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def _encode_sequence(self, x, lengths):
        B, T, C, H, W = x.shape
        frame_tokens = self.enc(x.view(B * T, C, H, W)).reshape(B, T, -1)
        packed = pack_padded_sequence(
            frame_tokens, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        key_padding_mask = (
            torch.arange(out.size(1), device=lengths.device)[None, :] >= lengths[:, None]
        )
        attn_out, _ = self.mha(out, out, out, key_padding_mask=key_padding_mask)
        attn_out = attn_out.masked_fill(key_padding_mask.unsqueeze(-1), 0.0)
        return attn_out.sum(dim=1) / lengths.unsqueeze(-1)

    def forward(self, x, lengths):
        seq_emb = self._encode_sequence(x, lengths)
        return self.head(seq_emb)

    @torch.inference_mode()
    def extract_embeddings(self, x, lengths, return_timewise=False):
        self.eval()
        seq_emb = self._encode_sequence(x, lengths)
        if return_timewise:
            attn_out = self._encode_sequence_with_attention(x, lengths)
            return seq_emb, attn_out
        return seq_emb

    def _encode_sequence_with_attention(self, x, lengths):
        B, T, C, H, W = x.shape
        frame_tokens = self.enc(x.view(B * T, C, H, W)).reshape(B, T, -1)
        packed = pack_padded_sequence(
            frame_tokens, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        key_padding_mask = (
            torch.arange(out.size(1), device=lengths.device)[None, :] >= lengths[:, None]
        )
        attn_out, _ = self.mha(out, out, out, key_padding_mask=key_padding_mask)
        return attn_out.masked_fill(key_padding_mask.unsqueeze(-1), 0.0)


In [ ]:
def collect_embeddings_with_paths(model, data_loader, device):
    ds = data_loader.dataset
    assert hasattr(ds, "folders"), "Dataset must expose .folders list of sequence directories."
    model.eval()
    X_list, y_list = [], []
    with torch.inference_mode():
        for batch in data_loader:
            if batch is None:
                continue
            seqs, lengths, labs = batch
            seqs = seqs.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            seq_emb = model.extract_embeddings(seqs, lengths)
            X_list.append(seq_emb.cpu().numpy())
            y_list.append(labs.numpy())
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    paths = list(ds.folders)
    assert len(paths) == len(X), f"Mismatch: {len(paths)} paths vs {len(X)} embeddings"
    return X, y, paths

def make_umap(X, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    um = umap.UMAP(
        n_neighbors=n_neighbors, min_dist=min_dist, metric=metric, random_state=random_state
    )
    return um.fit_transform(Xs)

def cluster_umap(Z, random_state=42):
    km = KMeans(n_clusters=3, n_init="auto", random_state=random_state)
    labels = km.fit_predict(Z)
    return labels, km.cluster_centers_

def pick_representatives(Z, cluster_labels, k=1):
    reps = {}
    for cluster_id in np.unique(cluster_labels):
        idx = np.where(cluster_labels == cluster_id)[0]
        pts = Z[idx]
        center = pts.mean(axis=0)
        dists = np.linalg.norm(pts - center, axis=1)
        reps[int(cluster_id)] = idx[np.argsort(dists)[:k]].tolist()
    return reps

def save_sequence_gif(seq_folder: Path, out_path: Path, size=IMG_SZ, fps=5, optimize=True):
    tif_paths = sorted(seq_folder.glob("*.tif"))
    if len(tif_paths) == 0:
        print(f"[WARN] No .tif files in {seq_folder}")
        return
    frames = []
    for p in tif_paths:
        im = Image.open(p).convert("RGB")
        w, h = im.size
        if w <= h:
            new_w = size
            new_h = int(h * size / w)
        else:
            new_h = size
            new_w = int(w * size / h)
        im = im.resize((new_w, new_h), resample=Image.BILINEAR)
        frames.append(im)
    duration_ms = int(1000 / fps)  
    out_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(out_path, save_all=True, append_images=frames[1:],
                   duration=duration_ms, loop=0, optimize=optimize)
    print(f"[GIF] Saved: {out_path}")

In [ ]:
model = BiLSTMSeqClassifier().to(DEVICE)
backbone_params = list(model.enc.backbone.parameters())
other_params = [p for n, p in model.named_parameters() if not n.startswith("enc.backbone")]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": other_params, "lr": 1e-4},
    ],
    weight_decay=1e-5,
)
total_steps = EPOCHS * len(train_dl)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[1e-5, 1e-4], total_steps=total_steps
)
criterion = nn.CrossEntropyLoss()
best_val_bal_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    for p in model.enc.backbone.parameters():
        p.requires_grad = epoch > 5

    model.train()
    epoch_loss = 0.0

    for batch in train_dl:
        if batch is None:
            continue
        seqs, lengths, labs = batch
        seqs = seqs.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        labs = labs.to(DEVICE, non_blocking=True)

        preds = model(seqs, lengths)
        loss = criterion(preds, labs)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item() * seqs.size(0)

    train_loss = epoch_loss / len(train_ds)
    model.eval()
    all_preds, all_labels, all_scores = [], [], []

    with torch.inference_mode():
        for seqs, lengths, labs in val_dl:
            if seqs is None or lengths is None or labs is None:
                continue
            seqs = seqs.to(DEVICE, non_blocking=True)
            lengths = lengths.to(DEVICE, non_blocking=True)
            logits = model(seqs, lengths)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1).cpu().numpy()
            prob_pos = probs[:, POS_IDX].cpu().numpy()
            labels = labs.cpu().numpy().astype(int)
            all_preds.append(preds)
            all_labels.append(labels)
            all_scores.append(prob_pos)

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_scores = np.concatenate(all_scores, axis=0)

    bal = balanced_accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    best_val_bal_acc = max(best_val_bal_acc, bal)

    print(f"Epoch {epoch:2d}/{EPOCHS} train_loss {train_loss:.4f} | val_bal_acc {bal:.3f} | recall {rec:.3f} | f1 {f1:.3f}")

    # in-training PR/ROC
    precision, recall, pr_thresholds = precision_recall_curve(all_labels, all_scores, pos_label=1)
    avg_prec = average_precision_score(all_labels, all_scores)
    
    fpr, tpr, roc_thresholds = roc_curve(all_labels, all_scores, pos_label=1)
    roc_auc = roc_auc_score(all_labels, all_scores)

    counter += 1
    if counter == 20:
        # Plot PR curve
        plt.figure(figsize=(6,5))
        plt.plot(recall, precision, label=f'AP = {avg_prec:.3f}')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision–Recall Curve')
        plt.legend()
        plt.grid(True)
        plt.show()

        # plot ROC curve
        plt.figure()
        plt.plot(fpr, tpr, label='ROC curve (area = {:.2f})'.format(roc_auc))
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc='lower right')
        plt.show()
        
        counter = 0

print(f"\nBest validation balanced accuracy: {best_val_bal_acc:.3f}")


In [ ]:
# ── UMAP, CLUSTERING & REPRESENTATIVES ON TRAIN+VAL ─────────────────────
model.eval()

train_eval_ds = SequenceDataset(train_folders, LABEL_MAP, val_tfms) 
train_eval_dl = DataLoader(
    train_eval_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,           
    collate_fn=collate_fn,
    num_workers=0
)
X_train, y_train, train_paths = collect_embeddings_with_paths(model, train_eval_dl, DEVICE)
X_val,   y_val,   val_paths   = collect_embeddings_with_paths(model, val_dl,       DEVICE)
X_all      = np.vstack([X_train, X_val])          
y_all      = np.concatenate([y_train, y_val])    
split_all  = np.array(["train"]*len(X_train) + ["val"]*len(X_val)) 
paths_all  = train_paths + val_paths           
folder_names_all = [Path(p).name for p in paths_all]

# UMAP projection (2D) on TRAIN+VAL
Z_all = make_umap(
    X_all, n_neighbors=15, min_dist=0.05,
    metric="cosine", random_state=42
)  
cluster_labels_all, centers_all = cluster_umap(Z_all, random_state=42)
plt.figure(figsize=(8,7))
colors = np.array(["tab:blue", "tab:orange", "tab:green", "tab:purple"])
for c in np.unique(cluster_labels_all):
    idx = np.where(cluster_labels_all == c)[0]
    plt.scatter(Z_all[idx, 0], Z_all[idx, 1], s=160, c=colors[c], alpha=0.75, label=f"Cluster {c}")
markers = np.array(["o", "x"])  # 0=serial-lt-hsc, 1=others
train_idx = np.where(split_all == "train")[0]
val_idx   = np.where(split_all == "val")[0]

for cls in np.unique(y_all):
    idx_train_cls = train_idx[y_all[train_idx] == cls]
    idx_val_cls   = val_idx[y_all[val_idx] == cls]

    if markers[cls] == "o":
        plt.scatter(Z_all[idx_val_cls, 0], Z_all[idx_val_cls, 1],
                    s=80, facecolors="none", edgecolors="k", marker="o",
                    linewidths=0.7, alpha=0.9, zorder=3, label=f"{CLASSES[cls]}")
    else:  
        plt.scatter(Z_all[idx_val_cls, 0], Z_all[idx_val_cls, 1],
                    s=80, color="k", marker="x",
                    linewidths=0.7, alpha=0.7, zorder=3, label=f"{CLASSES[cls]}")

plt.title("UMAP of Sequence Embeddings")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(loc="best", fontsize=8, ncol=2)
plt.grid(True, alpha=0.3) 
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1D UMAP
umap_model = UMAP(n_components=1, n_neighbors=15, min_dist=0.05,
                  metric="cosine", random_state=42)
Z_all = umap_model.fit_transform(X_all) 
Z_all_1d = Z_all.ravel()

cluster_labels_all, centers_all = cluster_umap(Z_all_1d.reshape(-1,1), n_clusters=3, random_state=42)
centers_all = centers_all.ravel() 
plt.figure(figsize=(10,3))
colors = np.array(["tab:blue", "tab:orange", "tab:green", "tab:purple"])
for c in np.unique(cluster_labels_all):
    idx = np.where(cluster_labels_all == c)[0]
    plt.scatter(Z_all_1d[idx], np.zeros_like(idx), s=160, c=colors[c], alpha=0.8, label=f"Cluster {c}")
markers = np.array(["o", "x"])
val_idx   = np.where(split_all == "val")[0]
y_offset = 0.12
for cls in np.unique(y_all):
    idx_train_cls = train_idx[y_all[train_idx] == cls]
    idx_val_cls   = val_idx[y_all[val_idx] == cls]
    # train points slightly above center line
    plt.scatter(Z_all_1d[idx_train_cls], np.full_like(idx_train_cls, y_offset),
                s=80, facecolors="none" if markers[cls]=="o" else "none",
                edgecolors="k" if markers[cls]=="o" else "k",
                marker=markers[cls], label=f"{CLASSES[cls]}")
    # val points slightly below center line
    plt.scatter(Z_all_1d[idx_val_cls], np.full_like(idx_val_cls, -y_offset),
                s=80, facecolors="none" if markers[cls]=="o" else "k",
                edgecolors="k" if markers[cls]=="o" else "k",
                marker=markers[cls], label=f"{CLASSES[cls]}")
plt.title("UMAP-1 (1D) of Sequence Embeddings")
plt.xlabel("UMAP-1")
plt.yticks([])          
plt.legend(loc="upper right", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# extract umap (1D) coordinates to file
df = pd.DataFrame({
    "umap1": Z_all_1d,                
    "label": y_all,                  
    "split": split_all,               
    "path": paths_all,               
    "folder_name": folder_names_all,  
    "cluster": cluster_labels_all    
})
df["path"] = df["path"].astype(str)
print(df)
df.to_csv("umap1d_embeddings.csv", index=False)

In [ ]:
# pick representative data points from umap
val_indices          = np.where(split_all == "val")[0]
Z_val_from_all       = Z_all[val_indices]                
cluster_labels_val   = cluster_labels_all[val_indices]   
val_paths_ordered    = [paths_all[i] for i in val_indices]
representatives = pick_representatives(Z_val_from_all, cluster_labels_val, k=1)

# Save GIFs for selected representatives
out_dir = Path("representatives_gifs")
for c, idx_list in representatives.items():
    for rank, idx_local in enumerate(idx_list, start=1):
        seq_dir = val_paths_ordered[idx_local]
        out_path = out_dir / f"cluster{c}_rep{rank}_{seq_dir.name}.gif"
        save_sequence_gif(seq_dir, out_path, size=IMG_SZ, fps=5)